# 04. 제조 데이터 Feature Engineering
**SK하이닉스 Autonomous R&D — Day 1**

---

## 우리가 다루는 데이터

반도체 공정에서 수집되는 데이터는 크게 세 종류입니다.

| 파일 | 내용 | 형태 |
|------|------|------|
| `Yield_Raw_Data.csv` | Lot이 어떤 공정 → 어떤 설비를 거쳤는지 이력 | 행: Lot×공정 |
| `Sensor_Data/LOT001.csv` | 공정 중 센서가 측정한 시계열 값 | 행: 시간(Step), 열: 센서 |
| `ImageData/LOT001_W01.csv` | Wafer 다이맵 (0=양품, 1=불량) | 6×6 행렬 |

## 우리의 목표

세 종류의 원본 데이터를 하나의 **분석 가능한 테이블**로 만드는 것입니다.

```
Yield_Raw_Data  ──→  Rework 제거  ──→  Route Table
                                      (Lot × 공정별 설비)
                                            │
Sensor_Data     ──→  요약통계량 추출  ──→  Sensor Feature
                     (mean/std/min/max)   (Lot × Feature)
                                            │
ImageData       ──→  Yield 계산  ────→  Wafer Table
                     (1 - 불량/전체)    (Wafer × Yield)
                                            │
                                      ┌─────┴─────┐
                                      │  Merge    │
                                      └─────┬─────┘
                                            ↓
                                   final_dataset.csv
                              (행: Wafer, 열: 27개 Feature + Y)
```



In [ ]:
# 데이터 경로 설정
import os
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/SK Autonomous R&D'

# dataset 폴더가 없으면 zip 압축 해제
if not os.path.exists(f'{BASE}/dataset'):
    if os.path.exists(f'{BASE}/dataset.zip'):
        print('dataset.zip 압축 해제 중...')
        os.system(f'unzip -q "{BASE}/dataset.zip" -d "{BASE}"')
        print('완료')
    else:
        print('⚠️  dataset.zip 을 SK Autonomous R&D 폴더에 업로드하세요.')

DATA_DIR = f'{BASE}/dataset'
print(f'DATA_DIR = {DATA_DIR}')


---
## Part 1. Route 데이터 — Rework 제거

### 데이터 구조
`Yield_Raw_Data.csv` 는 Lot이 공정을 거칠 때마다 한 행이 기록됩니다.

```
LotName  | Process | Equip_ID | Event_Time          | Y
---------|---------|----------|---------------------|---
LOT001   | 1700A   | E01      | 2024-01-15 08:30:00 | 76
LOT001   | 1700A   | E02      | 2024-01-15 10:10:00 | 76   ← Rework!
LOT001   | 1300A   | F01      | 2024-01-15 14:20:00 | 76
LOT002   | 1700A   | E01      | 2024-01-16 09:00:00 | 92
```

### 문제: Rework
LOT001이 1700A 공정을 **두 번** 거쳤습니다 (E01 → 불량 → 재처리 E02).  
이 경우 분석에 쓸 이력은 **마지막 처리 기록(E02)** 하나만 남겨야 합니다.

### 해결: `drop_duplicates(keep='last')`
같은 `(LotName, Process)` 조합이 여러 번 등장하면 마지막 행만 유지합니다.


In [ ]:
# Rework 확인


In [ ]:
# Rework 제거 — 마지막 이력만 유지


---
## Part 2. Route Table 생성

### 현재 형태 vs 목표 형태

Rework를 제거한 후에도 한 Lot의 이력이 **여러 행**에 걸쳐 있습니다.  
머신러닝 모델에 넣으려면 **한 Lot = 한 행** 형태가 필요합니다.

```
[ 현재 ]                          [ 목표 ]
LotName  Process  Equip_ID        LotName  1700A  1300A  4100A
LOT001   1700A    E02      →      LOT001   E02    F01    G03
LOT001   1300A    F01             LOT002   E01    F02    G01
LOT001   4100A    G03
LOT002   1700A    E01
LOT002   1300A    F02
LOT002   4100A    G01
```

### 해결: `pivot_table()`
공정(Process)을 컬럼으로 펼치고, 각 셀에 설비(Equip_ID)를 채웁니다.  
어떤 Lot이 어떤 공정에서 어떤 설비를 사용했는지 한눈에 볼 수 있습니다.


---
## Part 3. 센서 Feature 추출

### 데이터 구조
`Sensor_Data/` 폴더 안에 Lot별로 파일이 하나씩 있습니다.  
각 파일은 공정 중 센서가 시간(Step)에 따라 측정한 값입니다.

```
Step | Sensor_1 | Sensor_2 | Sensor_3 | Sensor_4
-----|----------|----------|----------|---------
0    | 101.2    | 198.4    | 49.7     | 302.1
1    | 102.5    | 201.1    | 50.2     | 298.7
2    | 99.8     | 197.3    | 48.9     | 305.4
...  | ...      | ...      | ...      | ...
```

### 문제: Step 수가 Lot마다 다름
어떤 Lot은 80 Step, 어떤 Lot은 120 Step — 그대로는 하나의 테이블로 합칠 수 없습니다.

### 해결: 요약 통계량으로 압축
각 센서의 `mean / std / min / max` 4개 값만 뽑으면  
모든 Lot을 같은 길이(센서 수 × 4개)의 Feature 벡터로 표현할 수 있습니다.

```
LOT001  Sensor_1_mean  Sensor_1_std  ...  Sensor_4_max
        101.2          1.3           ...  308.5
```


In [ ]:

# 단일 파일 확인


In [ ]:
# 전체 Lot 요약통계량 추출


---
## Part 4. Wafer Yield 계산

### 데이터 구조
`ImageData/` 폴더 안에 Wafer별로 파일이 하나씩 있습니다.  
파일명: `LOT001_W01.csv` (LOT001의 첫 번째 Wafer)

```
0  0  1  0  0  0
0  0  0  0  1  0
0  1  0  0  0  0     ← 6×6 다이맵
0  0  0  0  0  1        0 = 양품, 1 = 불량
1  0  0  0  0  0
0  0  0  1  0  0
```

### Yield 계산
```
Yield = 1 - (불량 다이 수 / 전체 다이 수)
      = 1 - (5 / 36)
      = 0.861  (86.1%)
```

### LotName 추출
파일명 `LOT001_W01` 에서 `_` 앞부분을 잘라내면 LotName을 얻을 수 있습니다.  
이후 Merge에서 Lot 단위로 데이터를 합칠 때 사용합니다.


---
## Part 5. 최종 Merge

지금까지 만든 세 테이블을 `LotName` 기준으로 합칩니다.

```
wafer_df        (168행 × 3열)   WaferName, Yield, LotName
    ↓ merge on LotName
route_table     (50행  × 9열)   LotName, 공정별 설비 5개, Y, Result
    ↓ merge on LotName
sensor_features (50행  × 17열)  LotName, 센서 통계량 16개
    ↓
final_df        (168행 × 27열)  분석 준비 완료
```

> **왜 168행?**  
> Route Table은 Lot 단위(50행)이지만 Wafer는 Lot당 여러 장이므로  
> Wafer 기준으로 merge하면 행이 늘어납니다.


---
## Part 6. 탐색 — 설비별 Yield 차이가 있는가?

최종 데이터셋이 완성됐습니다.  
간단한 분석 한 가지: **같은 공정이라도 어떤 설비를 쓰느냐에 따라 Yield가 달라지는가?**



In [ ]:
# 설비별 Yield


---
## ✅ Day 1 전체 흐름 정리

```
01 Python 기초     변수, 리스트, 딕셔너리, 함수, for, if     60분
02 NumPy + Maze    배열, 랜덤, 시각화, 알고리즘 사고          60분
03 Pandas          DataFrame, groupby, merge, EDA, 전처리   180분
   + 월드컵 과제 / 넷플릭스 과제
04 제조 데이터     Route→Rework 제거→Route Table             90분
                   Sensor Feature→Wafer Yield→최종 Merge
      ↓
Day 2~  Cursor / Claude Code / Agent
```

